# SMA Crossover (10-50) on SPY
## Strategy Brief
The SMA Crossover (10-50) strategy is a trend-following strategy that uses two simple moving averages (SMAs) to generate trading signals. The strategy predicts potential bullish or bearish trends based on the crossover of a short-term 10-day SMA and a long-term 50-day SMA. When the 10-day SMA crosses above the 50-day SMA, it signals a buy (long) opportunity, while a crossover below signals a sell (flat/short) opportunity. Historically, this strategy has shown potential to capture sustained market trends, but may underperform in sideways markets.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 — Trading Context
The hypothesis for the SMA Crossover (10-50) strategy is that when a short-term moving average crosses above a long-term moving average, it indicates a potential upward trend, and vice versa for a downward trend. All parameters for the strategy are set as constants for easy adjustments and clarity.

In [ ]:
TICKER = "SPY"
START_DATE = "2010-01-01"
END_DATE = "2023-10-01"
SHORT_WINDOW = 10
LONG_WINDOW = 50

## PHASE 2 — Data Exploration
In this phase, we will download historical price data for SPY using yfinance, compute the SMA Crossover (10-50) indicators, and visualize them overlaid on the price data to understand their behavior.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute SMAs
data['SMA10'] = data['Close'].rolling(window=SHORT_WINDOW).mean()
data['SMA50'] = data['Close'].rolling(window=LONG_WINDOW).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close Price', color='black')
plt.plot(data['SMA10'], label='10-Day SMA', color='blue', linestyle='--')
plt.plot(data['SMA50'], label='50-Day SMA', color='red', linestyle='--')
plt.title('SPY Price and SMA Crossover (10-50)')
plt.legend()
plt.show()

## PHASE 3 — Strategy Engineering
We will generate the trading signals based on the SMA crossover, define the entry and exit logic, and create a positions series to represent our trading stance (long, flat, or short).

In [ ]:
# Generate signals
data['Signal'] = 0
# Buy signal
data.loc[data['SMA10'] > data['SMA50'], 'Signal'] = 1
# Sell signal
data.loc[data['SMA10'] < data['SMA50'], 'Signal'] = -1

# Positions
positions = data['Signal'].copy()

## PHASE 4 — Coding & Backtesting
We will shift the signal to prevent look-ahead bias, compute the daily returns of the strategy versus a buy-and-hold approach, and visualize the equity curves of both strategies.

In [ ]:
# Prevent look-ahead bias
positions = positions.shift(1).fillna(0)

# Calculate returns
data['Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = positions * data['Returns']

# Calculate cumulative returns
data['Cumulative_Market'] = (1 + data['Returns']).cumprod()
data['Cumulative_Strategy'] = (1 + data['Strategy_Returns']).cumprod()

# Plot equity curves
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative_Market'], label='Market Returns', color='black')
plt.plot(data['Cumulative_Strategy'], label='Strategy Returns', color='blue')
plt.title('Equity Curve: Strategy vs. Market')
plt.legend()
plt.show()

## PHASE 5 — Performance Evaluation
We will evaluate the strategy's performance by calculating key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and max drawdown. We will also compare these metrics against the SPY buy-and-hold strategy.

In [ ]:
import numpy as np

# Performance metrics
def calculate_cagr(cumulative_returns):
    n_years = (cumulative_returns.index[-1] - cumulative_returns.index[0]).days / 365.25
    return (cumulative_returns[-1] ** (1 / n_years)) - 1

def calculate_sharpe(returns, risk_free_rate=0):
    return (returns.mean() - risk_free_rate) / returns.std() * np.sqrt(252)

def calculate_sortino(returns, risk_free_rate=0):
    downside_std = returns[returns < 0].std()
    return (returns.mean() - risk_free_rate) / downside_std * np.sqrt(252)

def calculate_max_drawdown(cumulative_returns):
    drawdown = cumulative_returns / cumulative_returns.cummax() - 1
    return drawdown.min()

def calculate_calmar(cagr, max_drawdown):
    return cagr / abs(max_drawdown)

# Calculate metrics
strategy_cagr = calculate_cagr(data['Cumulative_Strategy'])
market_cagr = calculate_cagr(data['Cumulative_Market'])

strategy_sharpe = calculate_sharpe(data['Strategy_Returns'])
market_sharpe = calculate_sharpe(data['Returns'])

strategy_sortino = calculate_sortino(data['Strategy_Returns'])
market_sortino = calculate_sortino(data['Returns'])

strategy_max_drawdown = calculate_max_drawdown(data['Cumulative_Strategy'])
market_max_drawdown = calculate_max_drawdown(data['Cumulative_Market'])

strategy_calmar = calculate_calmar(strategy_cagr, strategy_max_drawdown)
market_calmar = calculate_calmar(market_cagr, market_max_drawdown)

# Print results
print("Performance Metrics:")
print(f"Strategy CAGR: {strategy_cagr:.2%}")
print(f"Market CAGR: {market_cagr:.2%}")
print(f"Strategy Sharpe Ratio: {strategy_sharpe:.2f}")
print(f"Market Sharpe Ratio: {market_sharpe:.2f}")
print(f"Strategy Sortino Ratio: {strategy_sortino:.2f}")
print(f"Market Sortino Ratio: {market_sortino:.2f}")
print(f"Strategy Max Drawdown: {strategy_max_drawdown:.2%}")
print(f"Market Max Drawdown: {market_max_drawdown:.2%}")
print(f"Strategy Calmar Ratio: {strategy_calmar:.2f}")
print(f"Market Calmar Ratio: {market_calmar:.2f}")

## PHASE 6 — Deploy & Monitor
We will create a monitoring function to download the last 60 days of SPY data, compute the current SMA Crossover (10-50) signal, and determine if the strategy is currently in a long or flat/short position.

In [ ]:
def monitor_strategy():
    # Download last 60 days of SPY data
    recent_data = yf.download(TICKER, period='60d')
    
    # Compute recent SMAs
    recent_data['SMA10'] = recent_data['Close'].rolling(window=SHORT_WINDOW).mean()
    recent_data['SMA50'] = recent_data['Close'].rolling(window=LONG_WINDOW).mean()
    
    # Determine current signal
    if recent_data['SMA10'].iloc[-1] > recent_data['SMA50'].iloc[-1]:
        print("Current Position: Long")
    else:
        print("Current Position: Flat/Short")

# Run the monitoring function
monitor_strategy()